In [1]:
@app.command()
def synthesize(
    source_path: Path = typer.Option(RAW_DATA_DIR / "PSP_Jan_Feb_2019.xlsx", help="Quelle für Verteilungen"),
    n: int = typer.Option(200, min=1, help="Anzahl synthetischer Zeilen"),
    out_path: Path = typer.Option(PROCESSED_DATA_DIR / "synthetic.parquet", help="Ziel-Datei"),
):
    """
    Erzeugt einfache synthetische Daten via Bootstrap + leichtem Jitter (nur für Tests/Demos).
    """
    df = read_table(source_path)
    # Outcome/gebührenspezifische Spalten entfernen, falls vorhanden
    drop_cols = {"transaction_success", "success", "fee_successful", "fee_not_successful"}
    keep = [c for c in df.columns if c not in drop_cols]
    df = df[keep]

    samp = df.sample(n=n, replace=True, random_state=42).reset_index(drop=True)
    # leichter Jitter für numerische Spalten
    for c in samp.select_dtypes(include=[np.number]).columns:
        sd = samp[c].std(ddof=0)
        jitter = 0.01 * (sd if sd > 0 else 1.0)
        samp[c] = samp[c] + np.random.normal(0.0, jitter, size=len(samp))

    samp.to_parquet(out_path, index=False)
    typer.echo(f"Synthetische Daten gespeichert: {out_path} (n={n})")

SyntaxError: invalid syntax (3737097518.py, line 1)

In [5]:
# === Setup & Imports ===
import sys
from pathlib import Path
project_root = Path().resolve().parent  # ggf. anpassen
sys.path.insert(0, str(project_root))

import pandas as pd

from creditcard_psp.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR


df = pd.read_pickle(PROCESSED_DATA_DIR / "df.pkl")
print(df)

                     tmsp      country  amount  success         PSP  \
0     2019-01-01 00:01:11      Germany      89        0     UK_Card   
1     2019-01-01 00:01:17      Germany      89        1     UK_Card   
2     2019-01-01 00:02:49      Germany     238        0     UK_Card   
3     2019-01-01 00:03:13      Germany     238        1     UK_Card   
4     2019-01-01 00:04:33      Austria     124        0  Simplecard   
...                   ...          ...     ...      ...         ...   
50324 2019-02-28 23:45:39  Switzerland     415        0     UK_Card   
50325 2019-02-28 23:46:48      Austria      91        0     UK_Card   
50326 2019-02-28 23:47:04      Austria      91        0     UK_Card   
50327 2019-02-28 23:47:36      Austria      91        0     UK_Card   
50328 2019-02-28 23:48:19      Austria      91        1   Moneycard   

       3D_secured    card  fee_successful  fee_not_successful  weekday  ...  \
0               0    Visa             3.0                 1.0       